# MATEN/T MiniAI v1 — Colab GPU Training

**BY MATEN/T · M/ License v1.0**

This notebook trains **actual random-init 86,019,712-parameter** MiniAI weights using the 94 synthetic JSONL shards. **Choose Runtime → Change runtime type → GPU.** The model supports **2K context**, although lower training lengths reduce GPU memory. Upload the bundled ZIP to your Google Drive first. Weights are **not pre-trained** and this dataset is not sufficient for a dependable general chatbot.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
else: print('WARNING: no CUDA GPU assigned. Change runtime to GPU for practical training.')

In [ ]:
from pathlib import Path
import shutil
ZIP_PATH = Path('/content/drive/MyDrive/MATEN-T-MiniAI-v1-94Datasets-GitHub.zip')  # change only if your filename differs
EXTRACT_ROOT = Path('/content')
assert ZIP_PATH.is_file(), f'Upload the ZIP to Google Drive or change ZIP_PATH: {ZIP_PATH}'
shutil.unpack_archive(str(ZIP_PATH), str(EXTRACT_ROOT), 'zip')
PROJECT = EXTRACT_ROOT / 'MATEN-T-MiniAI-v1-Multilingual'
assert (PROJECT / 'miniai.py').exists(), f'Expected project at {PROJECT}'
print('Project:', PROJECT)

In [ ]:
import sys,subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt')])
subprocess.check_call([sys.executable,str(PROJECT/'miniai.py'),'info'])

## Train all domains

Checkpoints saved to your Google Drive. **No `--max-steps`** means a full epoch through 90% of all 94 shards. This can take substantial GPU time and may exceed free Colab sessions. For a 2K training window change `256` to `2048` only if VRAM suffices. Adjust `--batch-size` down and `--grad-accum` up for out-of-memory errors.
The 9.4M procedurally generated rows are mostly numeric/template variants; one epoch is extremely long and does not imply model competence. Save your own checkpoints to Drive.


In [ ]:
from pathlib import Path
import subprocess,sys
CHECKPOINT_DIR = Path('/content/drive/MyDrive/MATEN-T-MiniAI-checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)
command=[sys.executable,str(PROJECT/'miniai.py'),'train','--device','cuda','--epochs','1',
         '--batch-size','1','--grad-accum','8','--sequence-length','256',
         '--save-every','200','--log-every','20','--output',str(CHECKPOINT_DIR)]
print('Start training; leave notebook connected and monitor logs. Run resume cell after interruptions.')
subprocess.run(command,check=True)

## Resume only if training was interrupted

Use the **same** tokenizer, data, sequence-length, batch-size and accumulation. Only load your own checkpoint.

In [ ]:
resume=CHECKPOINT_DIR/'last.pt'
assert resume.exists(), 'No saved checkpoint yet'
subprocess.run(command + ['--resume',str(resume)],check=True)

## Simple generation test (after at least one checkpoint)

Output quality will be limited, especially after only a few optimizer steps.

In [ ]:
import sys,torch
sys.path.insert(0,str(PROJECT))
from miniai import loaded_model,load_tokenizer,infer
model,cfg=loaded_model(CHECKPOINT_DIR/'last.pt',torch.device('cuda'))
tok=load_tokenizer()
print(infer(model,cfg,tok,torch.device('cuda'),'What is machine learning?',64,0.7))